# WEMoE NLP: Colab pipeline

Runs the full reproduction on a Colab GPU (a T4 is enough):

1. Setup: Drive, clone, install, unit tests
2. Optional smoke run (a few minutes, numbers are meaningless)
3. Experiment 1: five single-task BERT experts
4. Baselines: Simple Averaging and Task Arithmetic (lambda chosen on dev)
5. Experiment 2: WEMoE + FSFT (3 seeds), k sweep, intermediate-only ablation
6. Experiment 4: parameters, storage, latency
7. Summary tables

Checkpoints and results are written to Google Drive. Every step skips work that is already done,
so after a disconnect just rerun the cells.

If the repository is private, add a Colab secret named `GITHUB_TOKEN` (fine-grained, read-only).
Never paste a token into a cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/wemoe-nlp'
PATHS = f'--set paths.checkpoint_dir={DRIVE_ROOT}/checkpoints --set paths.results_dir={DRIVE_ROOT}/results'
CFG = f'--config configs/default.yaml {PATHS}'

In [ ]:
import os
import subprocess

REPO = 'nguyenbaanh7779/wemoe-nlp'
BRANCH = 'main'
WORKDIR = '/content/wemoe-nlp'

if not os.path.exists(WORKDIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')  # only needed for a private repository
    except Exception:
        pass
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    result = subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, url, WORKDIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError('git clone failed: ' + result.stderr.replace(token or '<no token>', '***'))
    # keep the token out of .git/config
    subprocess.run(['git', '-C', WORKDIR, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'],
                   check=True)

%cd {WORKDIR}
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!python -m pytest -q

## Optional: smoke run

Tiny subsets and a few steps, to check the whole pipeline end to end. Outputs stay on the VM.

In [ ]:
SMOKE = '--config configs/smoke.yaml --set paths.checkpoint_dir=/content/smoke/checkpoints --set paths.results_dir=/content/smoke/results'
!python scripts/train_expert.py {SMOKE}
!python scripts/eval_baselines.py {SMOKE}
!python scripts/train_wemoe.py {SMOKE}
!python scripts/summarize_results.py --results-dir /content/smoke/results

## Experiment 1: single-task experts

One recipe for every task (lr 2e-5, batch 32, 3 epochs, max length 128, fp16). The best epoch is
chosen on the dev split carved from the training set. About 40-60 minutes on a T4.

In [ ]:
!python scripts/train_expert.py {CFG}

## Baselines: Simple Averaging and Task Arithmetic

Task Arithmetic lambda is chosen on dev. WEMoE uses the same lambda to initialise its routers.

In [ ]:
!python scripts/eval_baselines.py {CFG}

## Experiment 2: WEMoE + FSFT

Routers only, cross-entropy on k=256 labelled examples per task from the few-shot pool (never seen by
the experts), best step on dev, a single evaluation on the official validation split.

In [ ]:
for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.seed={seed}

## k sweep (Experiment 6) and MoE-scope ablation (intermediate only, as in the DANC code)

In [ ]:
for k in [16, 64]:
    for seed in [0, 1, 2]:
        !python scripts/train_wemoe.py {CFG} --set wemoe.k={k} --set wemoe.seed={seed}

for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.moe_scope=intermediate --set wemoe.seed={seed}

## Experiment 4: parameters, storage and latency

In [ ]:
!python scripts/count_params.py {CFG} --latency

## Summary

In [ ]:
!python scripts/summarize_results.py --results-dir {DRIVE_ROOT}/results